# The Excellency Vault — EPUB Builder
Fetches all posts from The Short Bear's Substack via the internal JSON API (no browser rendering needed), sorts oldest-first, and builds a valid EPUB.

Run each cell in order (play button or Shift+Enter).

In [ ]:
# Step 1 - Install dependencies
!pip install requests -q

In [ ]:
# Step 2 - Discover all posts via Substack internal API
import requests
import time

BASE = 'https://theshortbear.substack.com'
HEADERS = {'User-Agent': 'Mozilla/5.0 (epub-builder/1.0)'}

def get_all_posts():
    posts = []
    offset = 0
    limit = 25
    while True:
        url = f'{BASE}/api/v1/posts?limit={limit}&offset={offset}&sort=new'
        resp = requests.get(url, headers=HEADERS, timeout=20)
        resp.raise_for_status()
        batch = resp.json()
        if not batch:
            break
        posts.extend(batch)
        print(f'  Got {len(posts)} posts so far...')
        if len(batch) < limit:
            break
        offset += limit
        time.sleep(1)
    return posts

print('Fetching post list...')
all_posts = get_all_posts()
all_posts.sort(key=lambda p: p.get('post_date', ''))

print(f'\nFound {len(all_posts)} posts (oldest first):')
for p in all_posts:
    print(f"  {p.get('post_date','?')[:10]}  {p.get('title','(no title)')}")

In [ ]:
# Step 3 - Fetch full HTML body for each post via the /api/v1/posts/{slug} endpoint
import re

def clean_html(html):
    # Remove img tags entirely (broken in offline EPUB, not needed for TTS)
    html = re.sub(r'<img[^>]*/?>', '', html)
    # Remove empty paragraphs left behind
    html = re.sub(r'<p>\s*</p>', '', html)
    # Remove Substack subscribe/share buttons and promo divs
    html = re.sub(r'<div[^>]*class="[^"]*subscribe[^"]*"[^>]*>.*?</div>', '', html, flags=re.DOTALL)
    return html.strip()

chapters = []
for i, post in enumerate(all_posts):
    title = post.get('title', 'Untitled')
    subtitle = post.get('subtitle', '')
    date = post.get('post_date', '')[:10]
    slug = post.get('slug', '')
    print(f'[{i+1}/{len(all_posts)}] {title}...')
    try:
        url = f'{BASE}/api/v1/posts/{slug}'
        resp = requests.get(url, headers=HEADERS, timeout=20)
        resp.raise_for_status()
        data = resp.json()
        html = clean_html(data.get('body_html', '') or '')
        if not html.strip():
            print('  (no body_html - placeholder will be used)')
    except Exception as e:
        print(f'  WARNING: {e}')
        html = ''
    chapters.append({'title': title, 'subtitle': subtitle, 'date': date, 'slug': slug, 'html': html})
    time.sleep(1.5)

print(f'\nDone. {len(chapters)} chapters ready.')
for ch in chapters:
    print(f"  {ch['date']}  {ch['title'][:50]}  ({len(ch['html'])} chars)")


In [ ]:
# Step 4 - Build EPUB as a zip file
import zipfile
import uuid
from datetime import datetime

OUTPUT = 'excellency_vault.epub'
BOOK_ID = str(uuid.uuid4())
NOW = datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ')

CSS = '''
body { font-family: Georgia, serif; line-height: 1.7; margin: 2em; color: #222; }
h1 { font-size: 1.8em; margin-bottom: 0.2em; }
h2 { font-size: 1.2em; color: #555; margin-top: 0; font-weight: normal; }
.date { color: #999; font-size: 0.85em; margin-bottom: 2em; display: block; }
blockquote { border-left: 3px solid #ccc; margin-left: 0; padding-left: 1.2em; color: #555; font-style: italic; }
hr { border: none; border-top: 1px solid #ddd; margin: 2em 0; }
img { max-width: 100%; height: auto; }
p { margin: 0.8em 0; }
'''

def make_chapter_html(title, subtitle, date, content_html):
    if not content_html.strip():
        content_html = '<p><em>Content could not be retrieved for this post.</em></p>'
    sub = f'<h2>{subtitle}</h2>' if subtitle else ''
    return f'''<?xml version="1.0" encoding="utf-8"?>
<!DOCTYPE html>
<html xmlns="http://www.w3.org/1999/xhtml" xml:lang="en">
<head>
<meta charset="utf-8"/>
<title>{title}</title>
<link rel="stylesheet" type="text/css" href="../styles/style.css"/>
</head>
<body>
<h1>{title}</h1>
{sub}
<span class="date">{date}</span>
<hr/>
{content_html}
</body>
</html>'''

manifest_items = []
spine_items = []
toc_items = []
chapter_files = []

for i, ch in enumerate(chapters):
    fname = f'ch{i+1:03d}.xhtml'
    item_id = f'ch{i+1:03d}'
    html = make_chapter_html(ch['title'], ch.get('subtitle',''), ch['date'], ch['html'])
    chapter_files.append((fname, html))
    manifest_items.append(f'<item id="{item_id}" href="text/{fname}" media-type="application/xhtml+xml"/>')
    spine_items.append(f'<itemref idref="{item_id}"/>')
    toc_items.append((item_id, fname, ch['title']))

manifest_str = '\n    '.join(manifest_items)
spine_str = '\n    '.join(spine_items)

nav_links = '\n'.join(
    f'      <li><a href="text/{fname}">{title}</a></li>'
    for _, fname, title in toc_items
)
nav_html = f'''<?xml version="1.0" encoding="utf-8"?>
<!DOCTYPE html>
<html xmlns="http://www.w3.org/1999/xhtml" xmlns:epub="http://www.idpf.org/2007/ops" xml:lang="en">
<head><meta charset="utf-8"/><title>Table of Contents</title></head>
<body>
<nav epub:type="toc" id="toc">
  <h1>The Excellency Vault</h1>
  <ol>
{nav_links}
  </ol>
</nav>
</body>
</html>'''

ncx_points = '\n'.join(
    f'''  <navPoint id="{item_id}" playOrder="{j+1}">
    <navLabel><text>{title}</text></navLabel>
    <content src="text/{fname}"/>
  </navPoint>'''
    for j, (item_id, fname, title) in enumerate(toc_items)
)
ncx = f'''<?xml version="1.0" encoding="utf-8"?>
<ncx xmlns="http://www.daisy.org/z3986/2005/ncx/" version="2005-1">
<head><meta name="dtb:uid" content="{BOOK_ID}"/></head>
<docTitle><text>The Excellency Vault</text></docTitle>
<navMap>
{ncx_points}
</navMap>
</ncx>'''

opf = f'''<?xml version="1.0" encoding="utf-8"?>
<package xmlns="http://www.idpf.org/2007/opf" version="3.0" unique-identifier="bookid">
<metadata xmlns:dc="http://purl.org/dc/elements/1.1/">
  <dc:identifier id="bookid">{BOOK_ID}</dc:identifier>
  <dc:title>The Excellency Vault</dc:title>
  <dc:creator>THE SHORT BEAR</dc:creator>
  <dc:language>en</dc:language>
  <meta property="dcterms:modified">{NOW}</meta>
</metadata>
<manifest>
  <item id="nav" href="nav.xhtml" media-type="application/xhtml+xml" properties="nav"/>
  <item id="ncx" href="toc.ncx" media-type="application/x-dtbncx+xml"/>
  <item id="css" href="styles/style.css" media-type="text/css"/>
  {manifest_str}
</manifest>
<spine toc="ncx">
  <itemref idref="nav"/>
  {spine_str}
</spine>
</package>'''

with zipfile.ZipFile(OUTPUT, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.writestr('mimetype', 'application/epub+zip', compress_type=zipfile.ZIP_STORED)
    zf.writestr('META-INF/container.xml', '''<?xml version="1.0"?>
<container version="1.0" xmlns="urn:oasis:names:tc:opendocument:xmlns:container">
  <rootfiles>
    <rootfile full-path="OEBPS/content.opf" media-type="application/oebps-package+xml"/>
  </rootfiles>
</container>''')
    zf.writestr('OEBPS/content.opf', opf)
    zf.writestr('OEBPS/nav.xhtml', nav_html)
    zf.writestr('OEBPS/toc.ncx', ncx)
    zf.writestr('OEBPS/styles/style.css', CSS)
    for fname, html in chapter_files:
        zf.writestr(f'OEBPS/text/{fname}', html)

print(f'EPUB saved: {OUTPUT}  ({len(chapters)} chapters)')

In [ ]:
# Step 5 - Download to your device
from google.colab import files
files.download(OUTPUT)
print('Download started!')